# Phase 1: Marengo Video Embeddings

Generate 512-dimensional multimodal embeddings for video samples using the **Twelve Labs Marengo 3.0** model.

**Pipeline:** Load dataset &rarr; Select balanced subset across categories &rarr; Upload each video to Twelve Labs &rarr; Generate fused asset-level embedding &rarr; Store in FiftyOne

**Key specs:**
- Model: `marengo3.0` (V2 Embed API)
- Output: 512-d float vector per video (fused visual + audio)
- Dataset: [Voxel51/Safe_and_Unsafe_Behaviours](https://huggingface.co/datasets/Voxel51/Safe_and_Unsafe_Behaviours)
- Sampling: 30 videos, balanced across categories (15 per category)

## 1. Setup & Imports

In [ ]:
import os
import time
from collections import Counter

import numpy as np
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub
from twelvelabs import TwelveLabs, VideoInputRequest, MediaSource

# Configuration
MAX_SAMPLES = 30
DATASET_NAME = "Voxel51/Safe_and_Unsafe_Behaviours"

# Validate API key
assert os.environ.get("TWELVELABS_API_KEY"), (
    "Set TWELVELABS_API_KEY env var before running this notebook"
)

client = TwelveLabs(api_key=os.environ["TWELVELABS_API_KEY"])
print("Twelve Labs client initialized")
print(f"Target: {MAX_SAMPLES} samples from {DATASET_NAME}")

## 2. Load Dataset

Load the full dataset from FiftyOne (or HuggingFace if first run), then select a balanced subset across categories.

In [ ]:
# Load or reuse existing dataset
if fo.dataset_exists(DATASET_NAME):
    dataset = fo.load_dataset(DATASET_NAME)
    print(f"Loaded existing dataset: {DATASET_NAME}")
else:
    dataset = load_from_hub(DATASET_NAME)
    print(f"Downloaded from HuggingFace: {DATASET_NAME}")

print(f"Total samples: {len(dataset)}")
print(f"Media type: {dataset.media_type}")

# Show category distribution
all_labels = [s.ground_truth.label for s in dataset if s.ground_truth]
print(f"\nCategory distribution:")
for label, count in Counter(all_labels).most_common():
    print(f"  {label}: {count}")

## 3. Inspect a Sample

Look at one video sample to understand the data structure before embedding.

In [4]:
sample = dataset.first()
print(f"Filepath: {sample.filepath}")
print(f"File size: {os.path.getsize(sample.filepath) / 1024:.1f} KB")
print(f"Existing fields: {list(sample.field_names)}")
print(sample)

Filepath: /Users/rishimule/fiftyone/huggingface/hub/Voxel51/Safe_and_Unsafe_Behaviours/data/0_tr1.mp4
File size: 25259.7 KB
Existing fields: ['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'ground_truth']
<Sample: {
    'id': '6979231a91651488c3f85b00',
    'media_type': 'video',
    'filepath': '/Users/rishimule/fiftyone/huggingface/hub/Voxel51/Safe_and_Unsafe_Behaviours/data/0_tr1.mp4',
    'tags': ['train'],
    'metadata': <VideoMetadata: {
        'size_bytes': 25865947,
        'mime_type': 'video/mp4',
        'frame_width': 1920,
        'frame_height': 1080,
        'frame_rate': 24.83,
        'total_frame_count': 348,
        'duration': 14.015304,
        'encoding_str': 'avc1',
    }>,
    'created_at': datetime.datetime(2026, 4, 3, 16, 32, 27, 878000),
    'last_modified_at': datetime.datetime(2026, 4, 3, 16, 32, 27, 878000),
    'ground_truth': <Classification: {
        'id': '6979231a91651488c3f85aff',
        'tags': [],
        'label': 'Safe

## 4. Select Balanced Subset

Take an equal number of samples from each category to ensure the embedding set has diverse representation.

In [ ]:
categories = list(Counter(all_labels).keys())
n_categories = len(categories)
per_category = MAX_SAMPLES // n_categories

selected_ids = []
for cat in categories:
    view = dataset.match({"ground_truth.label": cat}).limit(per_category)
    selected_ids.extend([s.id for s in view])
    print(f"  {cat}: {len(view)} samples selected")

subset = dataset.select(selected_ids)
print(f"\nSubset size: {len(subset)} ({per_category} per category)")

## 5. Generate Marengo Embeddings

For each video in the balanced subset:
1. **Check** if already embedded (skip if so)
2. **Upload** the local file to Twelve Labs via `client.assets.create()`
3. **Embed** using the V2 API with fused visual+audio at asset scope
4. **Store** the 512-d vector as `sample["embedding"]`

In [ ]:
success_count = 0
fail_count = 0
skip_count = 0
timings = []

for i, sample in enumerate(subset, start=1):
    filepath = sample.filepath
    filename = os.path.basename(filepath)

    # Skip if already embedded
    try:
        if sample["embedding"] is not None:
            skip_count += 1
            print(f"[{i}/{len(subset)}] {filename} — already embedded, skipping")
            continue
    except (KeyError, AttributeError):
        pass

    print(f"[{i}/{len(subset)}] {filename} ... ", end="", flush=True)

    start = time.time()
    try:
        # Upload video
        with open(filepath, "rb") as f:
            asset = client.assets.create(method="direct", file=f)

        # Generate fused asset-level embedding
        response = client.embed.v_2.create(
            input_type="video",
            model_name="marengo3.0",
            video=VideoInputRequest(
                media_source=MediaSource(asset_id=asset.id),
                embedding_option=["visual", "audio"],
                embedding_scope=["asset"],
                embedding_type=["fused_embedding"],
            ),
        )

        embedding = response.data[0].embedding
        sample["embedding"] = embedding
        sample.save()

        elapsed = time.time() - start
        timings.append(elapsed)
        success_count += 1
        print(f"{len(embedding)}-d  ({elapsed:.1f}s)")

    except Exception as e:
        elapsed = time.time() - start
        timings.append(elapsed)
        fail_count += 1
        print(f"FAILED ({elapsed:.1f}s): {e}")

dataset.save()
print(f"\nDone: {success_count} succeeded, {fail_count} failed, {skip_count} skipped")

## 6. Verify Embeddings

Confirm the stored embeddings have the correct shape and inspect their statistics.

In [ ]:
# Collect all embeddings from the subset into a matrix
embeddings = []
for sample in subset:
    try:
        emb = sample["embedding"]
        if emb is not None:
            embeddings.append(emb)
    except (KeyError, AttributeError):
        pass

embeddings = np.array(embeddings)
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Value range: [{embeddings.min():.4f}, {embeddings.max():.4f}]")
print(f"Mean norm: {np.linalg.norm(embeddings, axis=1).mean():.4f}")
print(f"Samples with embeddings: {len(embeddings)} / {len(subset)}")

## 7. Pairwise Similarity

Compute cosine similarity between all video pairs to see how the embeddings separate content.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(embeddings)

# Summary stats (excluding self-similarity diagonal)
mask = ~np.eye(len(embeddings), dtype=bool)
off_diag = sim_matrix[mask]
print(f"Pairwise cosine similarity (off-diagonal):")
print(f"  Min:  {off_diag.min():.4f}")
print(f"  Mean: {off_diag.mean():.4f}")
print(f"  Max:  {off_diag.max():.4f}")
print(f"\n{len(embeddings)} videos embedded across {n_categories} categories")

## 8. Timing Summary

How long did each embedding take? This helps estimate cost for larger datasets.

In [ ]:
if timings:
    timings_arr = np.array(timings)
    print(f"Total time:   {timings_arr.sum():.1f}s")
    print(f"Mean / video: {timings_arr.mean():.1f}s")
    print(f"Min:          {timings_arr.min():.1f}s")
    print(f"Max:          {timings_arr.max():.1f}s")
    print(f"\nEstimate for 100 videos: ~{timings_arr.mean() * 100 / 60:.0f} min")
    print(f"Estimate for 600 videos: ~{timings_arr.mean() * 600 / 60:.0f} min")
else:
    print("All samples were already embedded — no new timing data")

## 9. Launch FiftyOne App

Visualize the dataset in the FiftyOne App to browse videos and confirm the embedding field is populated.

In [10]:
session = fo.launch_app(dataset)


Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.14.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|

